# Phasor analysis of a LINCam recording with PhasorPy

A LINCam records every photon with its position and its arrival after the
laser pulse. The `photonscore` package reads the `.photons` file, and from the
first histogram on everything is [PhasorPy](https://www.phasorpy.org): no
fitting, no model chosen in advance, only the Fourier coefficients of each
pixel's decay and what the phasor plot makes of them.

The sample is a section of *Convallaria majalis* (lily of the valley), the
usual test slide for FLIM.

Six steps, all in [`phasor.py`](phasor.py), which this notebook only calls:

1. histogram the photons into a (y, x, time) cube;
2. the phasor of every pixel, calibrated against the instrument response;
3. drop dim pixels and median-filter the rest;
4. pick the harmonic;
5. the populations on the phasor plot, and where they are in the image;
6. two lifetimes and their share in every pixel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from photonscore import Vault

import phasor

FILE = "convallaria.photons"
PIXELS = 512   # pixels a side
BINS = 256     # time bins over the 4096 TAC channels

vault = Vault(FILE)
print(vault)
positions, channels, channel_ps = phasor.geometry(vault)
period_ns = channels * channel_ps * 1e-3
print(f"{channel_ps} ps a channel: the {channels} channels span "
      f"{period_ns:.2f} ns, a first harmonic of {1e3 / period_ns:.2f} MHz")

## 1. Photons into a cube

One pass over the file, in chunks, fills the (y, x, time) cube and the decay
of all photons at the file's own 12.3 ps a channel. Images are drawn the way
Photonscore Preview draws them.

In [ ]:
cube, decay = phasor.read(vault, PIXELS, BINS)
vault.close()
counts = cube.sum(axis=-1)
print(f"{int(counts.sum()):,} photons, cube {cube.shape}, "
      f"{cube.nbytes / 2**20:.0f} MB")

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 5.5),
                                  layout="constrained")
phasor.intensity_panel(left, counts)
t = np.arange(channels) * channel_ps * 1e-3
right.semilogy(t, decay)
right.set_xlabel("arrival after the laser pulse, ns")
right.set_ylabel("photons per channel")
right.set_title("All photons")

## 2. Phasors, and the reference they need

`phasor_from_signal` takes the Fourier coefficients of each pixel's decay:
`real` and `imag`, one plane per harmonic, and `mean`, photons per time bin.

Uncalibrated, they are rotated by wherever the laser pulse sits in the TAC
range, 10.5 ns here. The reference that undoes it is the instrument response:
some 80 ps wide, far narrower than a period, so its phasor is a point on the
unit circle at the phase of time zero, a reference of lifetime 0 for
`phasor_calibrate`. Time zero is where the decay of all photons climbs
fastest.

In [ ]:
from phasorpy.phasor import phasor_from_signal
from phasorpy.plot import PhasorPlot

t0 = phasor.time_zero(decay, channel_ps)
print(f"time zero at {t0:.3f} ns")

raw_mean, raw_real, raw_imag = phasor_from_signal(
  cube, axis=-1, harmonic=list(phasor.HARMONICS))
mean, real, imag = phasor.phasors(cube, period_ns, t0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), layout="constrained")
bright = raw_mean * BINS > 300
for ax, g, s, title in ((axes[0], raw_real[0], raw_imag[0], "as measured"),
                        (axes[1], real[0], imag[0], "calibrated")):
  plot = PhasorPlot(ax=ax, allquadrants=True, title=title)
  plot.hist2d(g[bright], s[bright], cmap="Greys", bins=300)
  plot.semicircle()

## 3. Dim pixels out, the rest filtered

Pixels with fewer than 300 photons are set to NaN, and a 3 × 3 median filter,
applied twice, calms the rest: a pixel of 300 photons scatters its phasor by a
few hundredths.

In [ ]:
MIN_PHOTONS = 300

del cube
mean, real, imag = phasor.clean(mean, real, imag, BINS, MIN_PHOTONS)
print(f"{int(np.isfinite(real[0]).sum()):,} pixels kept")

## 4. Which harmonic

The histogram is one period of a periodic signal, so every harmonic of
19.8 MHz is a phasor of its own. Lifetimes of 0.5 to 3 ns crowd near (1, 0) at
the first; at the third, 59.5 MHz, they spread over most of the semicircle.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), layout="constrained")
for ax, n in zip(axes, phasor.HARMONICS):
  h = phasor.HARMONICS.index(n)
  phasor.phasor_panel(ax, real[h], imag[h], n * 1e3 / period_ns,
                      harmonic=n)

## 5. Populations on the phasor plot

A Gaussian mixture finds the dense regions of the plot, and the ellipses it
returns are cursors: every pixel inside one gets its colour. A pixel inside two
goes to the nearer.

In [ ]:
HARMONIC = 3
CLUSTERS = 3

from phasorpy.lifetime import phasor_to_apparent_lifetime

h = phasor.HARMONICS.index(HARMONIC)
frequency = HARMONIC * 1e3 / period_ns
ellipses, masks = phasor.clusters(real[h], imag[h], CLUSTERS)
for i, (g, s) in enumerate(zip(ellipses[0], ellipses[1])):
  tau_phase, tau_mod = phasor_to_apparent_lifetime(g, s, frequency)
  print(f"cluster {i + 1}: tau_phase {float(tau_phase):.2f} ns, "
        f"tau_mod {float(tau_mod):.2f} ns, {int(masks[i].sum()):,} pixels")

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 6),
                                  layout="constrained")
phasor.phasor_panel(left, real[h], imag[h], frequency, ellipses,
                    harmonic=HARMONIC)
phasor.clusters_panel(right, counts, masks)

Every cluster lies inside the semicircle: no pixel decays with a single
lifetime, and the phase lifetime of each stays below its modulation lifetime,
as it must for a mixture.

## 6. Two components

If every pixel held the same two lifetimes in different proportions, its
phasor would lie on the line between them. The cloud's long axis meets the
semicircle at those two lifetimes, and a pixel's place along the line is the
share of the photons from each.

In [ ]:
lifetimes, fraction = phasor.components(mean, real[h], imag[h], frequency)
print(f"{lifetimes[0]:.2f} ns and {lifetimes[1]:.2f} ns")

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 6),
                                  layout="constrained")
phasor.phasor_panel(left, real[h], imag[h], frequency, lifetimes=lifetimes,
                    harmonic=HARMONIC)
phasor.fraction_panel(right, counts, fraction, lifetimes)

The cloud is wider than a line: more than two lifetimes are at work, and
the share is a summary of each pixel, not a fit of it.

## 7. The phasors, saved

An OME-TIFF of `mean`, `real` and `imag` for the three harmonics, with the
frequency: `phasorpy.io.phasor_from_ometiff` reads it back, calibrated.

In [ ]:
from phasorpy.io import phasor_from_ometiff, phasor_to_ometiff

phasor_to_ometiff("convallaria.phasor.ome.tif", mean, real, imag,
                  frequency=1e3 / period_ns,
                  harmonic=list(phasor.HARMONICS), dtype=np.float32)
m, g, s, attrs = phasor_from_ometiff("convallaria.phasor.ome.tif",
                                     harmonic="all")
print(g.shape, attrs["frequency"], attrs["harmonic"])